In [2]:
import numpy as np

emb = np.load("data/embeddings/image_urls_new.npy", allow_pickle=True)


In [7]:
len(set(emb))
len(emb)

20100

In [1]:
url = f"https://www.vogue.com/fashion-shows/fall-2010-ready-to-wear/oscar-de-la-renta"
print(f"Fetching: {url}")

import requests
import json
import re
from bs4 import BeautifulSoup
from unidecode import unidecode
import pandas as pd
import os
import sys

# Send request
r = requests.get(url)
if r.status_code != 200:
    print(f"Failed to retrieve {url}")


# Parse the page content
soup = BeautifulSoup(r.content, 'html.parser')


script_tag = soup.find("script", string=re.compile(r'"runwayShowGalleries":'))
script_content = script_tag.string if script_tag else None

def modify_image_url(original_url):
    # Replace the width parameter in the URL for a higher-resolution image
    return original_url.replace("w_360", "w_1280")
if script_content:
    json_data_match = re.search(r'"runwayShowGalleries":\s*({.*?})\s*;', script_content, re.DOTALL)
    if json_data_match:
        json_data_str = json_data_match.group(1).replace("\\u002F", "/")
        json_decoder = json.JSONDecoder()
        json_data, _ = json_decoder.raw_decode(json_data_str)

        galleries = json_data["galleries"]

        # Filter only gallery with title == "Collection"
        collection_gallery = [g for g in galleries if g.get("title") == "Collection"]

        images_data = []
        if collection_gallery:
            gallery = collection_gallery[0]  # should be only one
            for item in gallery["items"]:
                image_info = item.get("image", {})
                src = image_info.get("sources", {}).get("sm", {}).get("url")
                alt_text = image_info.get("altText")

                # Remove "Image may contain" if present
                if alt_text and alt_text.lower().startswith("image may contain"):
                    alt_text = alt_text[len("Image may contain"):].strip()

                if src:
                    images_data.append({
                        "url": modify_image_url(src),
                        "alt_text": alt_text, 
                    })
        else:
            images_data = []
            print("No Collection gallery found.")

Fetching: https://www.vogue.com/fashion-shows/fall-2010-ready-to-wear/oscar-de-la-renta


In [7]:
df = pd.read_parquet("data/data_vogue_final_with_images.parquet")

In [12]:
df

,fashion_house,show,URL,cover_image_url,year,category,season,location,description,editor,publish_date,old_designer_name,image_urls,image_urls_sample,annotation,creative_director,designer_name,double_cd,image_data
0,3 1 Phillip Lim,fall-2006-ready-to-wear,https://www.vogue.com/fashion-shows/fall-2006-...,https://assets.vogue.com/photos/55c6516608298d...,2006,ready-to-wear,fall,,While violinists played classic Prince hits in...,Laird Borrelli-Persson,"February 6, 2006",Phillip Lim,[https://assets.vogue.com/photos/55c6516608298...,[https://assets.vogue.com/photos/55c6516608298...,0,Phillip Lim,Phillip Lim,0,[{'alt_text': 'Clothing Apparel Human Person O...
1,3 1 Phillip Lim,fall-2007-ready-to-wear,https://www.vogue.com/fashion-shows/fall-2007-...,https://assets.vogue.com/photos/55c6517708298d...,2007,ready-to-wear,fall,,"""Clothes people wear."" With that deceptively s...",Nicole Phelps,"February 3, 2007",Phillip Lim,[https://assets.vogue.com/photos/55c6517708298...,[https://assets.vogue.com/photos/55c6517708298...,1,Phillip Lim,Phillip Lim,0,[{'alt_text': 'Clothing Apparel Human Person C...
2,3 1 Phillip Lim,spring-2007-ready-to-wear,https://www.vogue.com/fashion-shows/spring-200...,https://assets.vogue.com/photos/55c6517108298d...,2007,ready-to-wear,spring,,Phillip Lim's first-ever runway presentation g...,Nicole Phelps,"September 9, 2006",Phillip Lim,[https://assets.vogue.com/photos/55c6517108298...,[https://assets.vogue.com/photos/55c6517108298...,1,Phillip Lim,Phillip Lim,0,[{'alt_text': 'Lee Heejoon Human Person Clothi...
3,3 1 Phillip Lim,fall-2008-ready-to-wear,https://www.vogue.com/fashion-shows/fall-2008-...,https://assets.vogue.com/photos/55c6518908298d...,2008,ready-to-wear,fall,,He's a designer who often dismisses his clothe...,Meenal Mistry,"February 5, 2008",Phillip Lim,[https://assets.vogue.com/photos/55c6518908298...,[https://assets.vogue.com/photos/55c6518908298...,1,Phillip Lim,Phillip Lim,0,[{'alt_text': 'Human Person Clothing Apparel C...
4,3 1 Phillip Lim,spring-2008-ready-to-wear,https://www.vogue.com/fashion-shows/spring-200...,https://assets.vogue.com/photos/55c6518008298d...,2008,ready-to-wear,spring,,A Mercer Street store! A children's line! Eyew...,Meenal Mistry,"September 8, 2007",Phillip Lim,[https://assets.vogue.com/photos/55c6518008298...,[https://assets.vogue.com/photos/55c6518008298...,0,Phillip Lim,Phillip Lim,0,[{'alt_text': 'Irina Lazareanu Human Person Cl...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14282,Zuhair Murad,pre-fall-2024,https://www.vogue.com/fashion-shows/pre-fall-2...,https://assets.vogue.com/photos/659beac6347b9d...,2024,,pre-fall,,"With his pre-fall collection, Zuhair Murad tun...",Tina Isaac-Goizé,"January 8, 2024",Zuhair Murad,[https://assets.vogue.com/photos/659beac6347b9...,[https://assets.vogue.com/photos/659beaef28f4e...,0,Zuhair Murad,Zuhair Murad,0,"[{'alt_text': '', 'url': 'https://assets.vogue..."
14283,Zuhair Murad,spring-2024-ready-to-wear,https://www.vogue.com/fashion-shows/spring-202...,https://assets.vogue.com/photos/651ad34a299e54...,2024,ready-to-wear,spring,,"For spring, Zuhair Murad moved cross-country f...",Tina Isaac-Goizé,"October 2, 2023",Zuhair Murad,[https://assets.vogue.com/photos/651ad34a299e5...,[https://assets.vogue.com/photos/651ad34a299e5...,1,Zuhair Murad,Zuhair Murad,0,"[{'alt_text': '', 'url': 'https://assets.vogue..."
14284,Zuhair Murad,resort-2024,https://www.vogue.com/fashion-shows/resort-202...,https://assets.vogue.com/photos/649aa1e5c1aeb0...,2024,,resort,,Zuhair Murad has stars in his eyes. One is Tay...,Tina Isaac-Goizé,"June 28, 2023",Zuhair Murad,[https://assets.vogue.com/photos/649aa1e5c1aeb...,[https://assets.vogue.com/photos/649aa1e5c1aeb...,1,Zuhair Murad,Zuhair Murad,0,"[{'alt_text': '', 'url': 'https://assets.vogue..."
14285,Zuhair Murad,spring-2025-ready-to-wear,https://www.vogue.com/fashion-shows/spring-202...,https://assets.vogue.com/photos/66fac981075851...,2025,ready-to-wear,spring,,The 

In [16]:

df["image_urls_collection"] = df["image_data"].apply(
    lambda lst: [d.get("url") for d in lst if isinstance(d, dict)])

In [21]:
def align_sample_with_collection(sample, collection):
    # Ensure lists

    
    sample = list(sample)
    collection = list(collection)

    # 1. Keep only overlapping URLs
    new_sample = [url for url in sample if url in collection]

    # 2. Fill missing slots with unused URLs from the collection
    missing_count = len(sample) - len(new_sample)

    if missing_count > 0:
        # URLs in the collection not already in the sample
        unused = [url for url in collection if url not in new_sample]

        # Take as many as needed
        replacement = unused[:missing_count]

        # Extend sample
        new_sample.extend(replacement)

    return new_sample

df["image_urls_sample_fixed"] = df.apply(
    lambda row: align_sample_with_collection(
        row["image_urls_sample"], row["image_urls_collection"]
    ),
    axis=1
)

In [25]:
def align_sample_with_collection(sample, collection):


    sample = list(sample)
    collection = list(collection)

    # 1. Keep only overlapping URLs
    new_sample = [url for url in sample if url in collection]

    # 2. Count how many need to be added
    missing_count = len(sample) - len(new_sample)

    replacements = []  # store newly added URLs

    if missing_count > 0:
        # URLs in the collection not already in the sample
        unused = [url for url in collection if url not in new_sample]

        # Take as many as needed
        replacements = unused[:missing_count]

        # Extend sample
        new_sample.extend(replacements)

    return new_sample, replacements

df[["image_urls_sample_fixed", "replacements"]] = df.apply(
    lambda row: pd.Series(
        align_sample_with_collection(
            row["image_urls_sample"], 
            row["image_urls_collection"]
        )
    ),
    axis=1
)

In [31]:
df.to_parquet("data/data_vogue_final_with_images_sampled.parquet")

In [27]:
all_replacements = [
    url
    for sublist in df["replacements"]
    if isinstance(sublist, list)
    for url in sublist
]

In [33]:
replaced_urls = set([
    url
    for sublist in df["replacements"]
    if isinstance(sublist, list)
    for url in sublist])

In [37]:
df = pd.read_parquet("data/data_vogue_final_with_images_sampled.parquet")
import numpy as np
# 1. Collect replacement URLs
replaced_urls = {
    url
    for sublist in df["replacements"]
    if isinstance(sublist, list)
    for url in sublist
}
URLS_PATH = "data/embeddings/image_urls.npy"
def load_existing_urls_npy(urls_path=URLS_PATH):
    """Load already processed image URLs from .npy file."""
    if not os.path.exists(urls_path):
        return set()
    urls_array = np.load(urls_path, allow_pickle=True)
    return set(urls_array)

# 2. Load already processed URLs
processed_urls = load_existing_urls_npy(URLS_PATH)

# 3. Determine which replacement URLs still need processing
to_process = replaced_urls - processed_urls

In [41]:
import re

def get_image_id(url):
    m = re.search(r"photos/([0-9a-f]+)/", url)
    return m.group(1) if m else url
processed_ids = {get_image_id(u) for u in processed_urls}
sample_ids    = {get_image_id(u) for u in replaced_urls}

to_process = [u for u in replaced_urls if get_image_id(u) not in processed_ids]

In [51]:
df = pd.read_parquet("data/data_vogue_final_with_images_sampled.parquet")
import numpy as np
# 1. Collect replacement URLs
replaced_urls = list(
    url
    for sublist in df["replacements"]
    if isinstance(sublist, list)
    for url in sublist
)

replaced_urls = set([
    url
    for sublist in df["replacements"]
    for url in sublist])

In [52]:
replaced_urls

{'https://assets.vogue.com/photos/5b982245eee1f52dc47cf844/master/w_1280,c_limit/00009-Baja-East-Spring-2019-Ready-To-Wear-photographer-Nio-Vardan.jpg',
 'https://assets.vogue.com/photos/5b3d89f8314b0a7ebbba6a09/master/w_1280,c_limit/givenchy-fall-2005-couture-00020h-eugenia-volodina.jpg',
 'https://assets.vogue.com/photos/57e80b7a9186bce754b3b641/master/w_1280,c_limit/_ARB0010.jpg',
 'https://assets.vogue.com/photos/55c651c508298d8be227a922/master/w_1280,c_limit/BOT_0147.jpg',
 'https://assets.vogue.com/photos/55c6514b08298d8be21ef6d4/master/w_1280,c_limit/100005776.jpg',
 'https://assets.vogue.com/photos/55c6518a08298d8be2237fdf/master/w_1280,c_limit/00030m.jpg',
 'https://assets.vogue.com/photos/68307d8c509371b319fc614e/master/w_1280,c_limit/00005-valentino-fall-2020-ready-to-wear-credit-gorunway-model-jill%20kortleve.jpg',
 'https://assets.vogue.com/photos/680a026255e7d4224ead0399/master/w_1280,c_limit/00090-bill-blass-spring-2002-ready-to-wear-caroline-ribeiro.jpg',
 'https://asse

In [8]:
import requests
import json
import re
from bs4 import BeautifulSoup
from unidecode import unidecode
import pandas as pd
import os
import sys
def extract_collection_image_urls(soup):
    """
    Extract only the COLLECTION image URLs from the Vogue runway page.
    """
    image_urls = []

    # Find all links pointing to the Collection slideshow
    collection_links = soup.find_all("a", href=True)

    for link in collection_links:
        href = link["href"]
        # Ensure it refers to the main collection slideshow
        if "/slideshow/collection" in href:
            # Inside this link (or its children), find IMG tags
            imgs = link.find_all("img")
            for img in imgs:
                src = img.get("src")
                if src and "assets.vogue.com/photos" in src:
                    image_urls.append(src)

    # Remove duplicates and return
    return list(dict.fromkeys(image_urls))

url = f"https://www.vogue.com/fashion-shows/fall-2010-ready-to-wear/oscar-de-la-renta"
print(f"Fetching: {url}")

# Send request
r = requests.get(url)
if r.status_code != 200:
    print(f"Failed to retrieve {url}")


# Parse the page content
soup = BeautifulSoup(r.content, 'html.parser')

collections = extract_collection_image_urls(soup)

Fetching: https://www.vogue.com/fashion-shows/fall-2010-ready-to-wear/oscar-de-la-renta


In [92]:
url = f"https://www.vogue.com/fashion-shows/fall-2010-ready-to-wear/oscar-de-la-renta"
print(f"Fetching: {url}")

# Send request
r = requests.get(url)
if r.status_code != 200:
    print(f"Failed to retrieve {url}")


# Parse the page content
soup = BeautifulSoup(r.content, 'html.parser')

Fetching: https://www.vogue.com/fashion-shows/fall-2010-ready-to-wear/oscar-de-la-renta


In [94]:
script_tag = soup.find("script", string=re.compile(r'"runwayShowGalleries":'))
script_content = script_tag.string if script_tag else None

def modify_image_url(original_url):
    # Replace the width parameter in the URL for a higher-resolution image
    return original_url.replace("w_360", "w_1280")
if script_content:
    json_data_match = re.search(r'"runwayShowGalleries":\s*({.*?})\s*;', script_content, re.DOTALL)
    if json_data_match:
        json_data_str = json_data_match.group(1).replace("\\u002F", "/")
        json_decoder = json.JSONDecoder()
        json_data, _ = json_decoder.raw_decode(json_data_str)

        galleries = json_data["galleries"]

        # Filter only gallery with title == "Collection"
        collection_gallery = [g for g in galleries if g.get("title") == "Collection"]

        images_data = []
        if collection_gallery:
            gallery = collection_gallery[0]  # should be only one
            for item in gallery["items"]:
                image_info = item.get("image", {})
                src = image_info.get("sources", {}).get("sm", {}).get("url")
                alt_text = image_info.get("altText")

                # Remove "Image may contain" if present
                if alt_text and alt_text.lower().startswith("image may contain"):
                    alt_text = alt_text[len("Image may contain"):].strip()

                if src:
                    images_data.append({
                        "url": modify_image_url(src),
                        "alt_text": alt_text, 
                    })
        else:
            images_data = []
            print("No Collection gallery found.")

In [91]:
images_data

[{'url': 'https://assets.vogue.com/photos/55c651ba08298d8be226dcab/master/w_1280,c_limit/00010fullscreen.jpg',
  'alt_text': 'Clothing Apparel Coat Overcoat Human and Person',
  'model': 'Freja Beha Erichsen',
  'photographer': 'Marcio Madeira',
  'photographer_agency': 'FirstView.com'},
 {'url': 'https://assets.vogue.com/photos/55c651ba08298d8be226dcac/master/w_1280,c_limit/00020fullscreen.jpg',
  'alt_text': 'Clothing Apparel Human Person Sleeve Military Military Uniform Long Sleeve and Female',
  'model': 'Freja Beha Erichsen',
  'photographer': 'Marcio Madeira',
  'photographer_agency': 'FirstView.com'},
 {'url': 'https://assets.vogue.com/photos/55c651ba08298d8be226dcad/master/w_1280,c_limit/00030fullscreen.jpg',
  'alt_text': 'Clothing Sleeve Apparel Coat Long Sleeve Dress Overcoat Human Female Person and Woman',
  'model': 'Freja Beha Erichsen',
  'photographer': 'Marcio Madeira',
  'photographer_agency': 'FirstView.com'},
 {'url': 'https://assets.vogue.com/photos/55c651ba08298d8

In [86]:
info_div = soup.find("contentType")
info_div

In [ ]:
script_tag = soup.find("script", string=re.compile(r'"runwayShowGalleries":'))
script_content = script_tag.string if script_tag else None
def modify_image_url(original_url):
    # Replace the width parameter in the URL for a higher-resolution image
    return original_url.replace("w_360", "w_1280")

def model_photographer(slide_url):
    """Fetch model and photographer from individual slideshow page"""
    if not slide_url.startswith("http"):
        slide_url = "https://www.vogue.com" + slide_url

    r = requests.get(slide_url)
    if r.status_code != 200:
        return None, None

    soup = BeautifulSoup(r.content, "html.parser")

    # Find the div containing model and photo credit
    info_div = soup.find("div", class_="ImageInfoSection-ccujeY")
    if not info_div:
        return None, None

    # Extract model name
    model_tag = info_div.find("p", class_=re.compile(r"ImageInfoModel"))
    model_name = model_tag.get_text(strip=True) if model_tag else None

    # Extract photographer
    photo_tag = info_div.find("p", class_=re.compile(r"ImageInfoPhotoCredit"))
    photographer = None
    if photo_tag:
        text = photo_tag.get_text(strip=True)
        # Remove "Photo:" prefix if present
        if text.lower().startswith("photo:"):
            photographer = text[len("photo:"):].strip()
        else:
            photographer = text
    model_name = model_name.replace("Model: ","")
    photographer, photographer_agency = photographer.split("/")
    return model_name, photographer.strip(), photographer_agency.strip()

if script_content:
    json_data_match = re.search(r'"runwayShowGalleries":\s*({.*?})\s*;', script_content, re.DOTALL)
    if json_data_match:
        json_data_str = json_data_match.group(1).replace("\\u002F", "/")
        json_decoder = json.JSONDecoder()
        json_data, _ = json_decoder.raw_decode(json_data_str)

        galleries = json_data["galleries"]

        # Filter only gallery with title == "Collection"
        collection_gallery = [g for g in galleries if g.get("title") == "Collection"]

        images_data = []
        if collection_gallery:
            gallery = collection_gallery[0]  # should be only one
            for item in gallery["items"]:
                model, photographer, photographer_agency = model_photographer(item["url"])
                print(model, photographer, photographer_agency )
                image_info = item.get("image", {})
                src = image_info.get("sources", {}).get("sm", {}).get("url")
                alt_text = image_info.get("altText")

                # Remove "Image may contain" if present
                if alt_text and alt_text.lower().startswith("image may contain"):
                    alt_text = alt_text[len("Image may contain"):].strip()

                if src:
                    images_data.append({
                        "url": modify_image_url(src),
                        "alt_text": alt_text, 
                        "model" : model,
                        "photographer": photographer, 
                        "photographer_agency": photographer_agency
                    })
        else:
            images_data = []
            print("No Collection gallery found.")

https://www.vogue.com/fashion-shows/fall-2010-ready-to-wear/oscar-de-la-renta/slideshow/collection#1
Freja Beha Erichsen Marcio Madeira FirstView.com
https://www.vogue.com/fashion-shows/fall-2010-ready-to-wear/oscar-de-la-renta/slideshow/collection#2
Freja Beha Erichsen Marcio Madeira FirstView.com
https://www.vogue.com/fashion-shows/fall-2010-ready-to-wear/oscar-de-la-renta/slideshow/collection#3
Freja Beha Erichsen Marcio Madeira FirstView.com
https://www.vogue.com/fashion-shows/fall-2010-ready-to-wear/oscar-de-la-renta/slideshow/collection#4
Freja Beha Erichsen Marcio Madeira FirstView.com
https://www.vogue.com/fashion-shows/fall-2010-ready-to-wear/oscar-de-la-renta/slideshow/collection#5
Freja Beha Erichsen Marcio Madeira FirstView.com
https://www.vogue.com/fashion-shows/fall-2010-ready-to-wear/oscar-de-la-renta/slideshow/collection#6
Freja Beha Erichsen Marcio Madeira FirstView.com
https://www.vogue.com/fashion-shows/fall-2010-ready-to-wear/oscar-de-la-renta/slideshow/collection#7

KeyboardInterrupt: 

In [77]:
import requests
import re
import json
from bs4 import BeautifulSoup

def modify_image_url(original_url):
    """Replace width for higher-resolution images"""
    return original_url.replace("w_360", "w_1280")

def scrape_collection_with_credits(collection_url):
    """
    Scrapes all images from a Vogue collection page, along with model and photographer info.

    Returns a list of dicts:
    [
        {
            "url": ...,
            "alt_text": ...,
            "model": ...,
            "photographer": ...,
            "photographer_agency": ...
        },
        ...
    ]
    """
    r = requests.get(collection_url)
    if r.status_code != 200:
        print(f"Failed to fetch {collection_url}")
        return []

    soup = BeautifulSoup(r.content, "html.parser")

    # Find the script containing runwayShowGalleries JSON
    script_tag = soup.find("script", string=re.compile(r'"runwayShowGalleries":'))
    if not script_tag:
        print("No runwayShowGalleries script found.")
        return []

    script_content = script_tag.string

    json_match = re.search(r'"runwayShowGalleries":\s*({.*?})\s*;', script_content, re.DOTALL)
    if not json_match:
        print("No runwayShowGalleries JSON found.")
        return []

    json_str = json_match.group(1).replace("\\u002F", "/")

    try:
        data = json.loads(json_str)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON: {e}")
        return []

    galleries = data.get("galleries", [])
    collection_gallery = next((g for g in galleries if g.get("title") == "Collection"), None)

    if not collection_gallery:
        print("No Collection gallery found.")
        return []

    images_data = []

    for item in collection_gallery["items"]:
        image_id = item.get("id")
        model = item.get("model")
        credit = item.get("desktopCredit") or item.get("mobileCredit") or ""
        photographer, photographer_agency = (c.strip() for c in credit.split("/", 1)) if "/" in credit else (credit, "")

        # Construct image URL (Vogue pattern)
        if image_id:
            url = f"https://assets.vogue.com/photos/{image_id}/master/w_360,c_limit/00010fullscreen.jpg"
            url = modify_image_url(url)
        else:
            url = None

        alt_text = item.get("image", {}).get("altText")
        if alt_text and alt_text.lower().startswith("image may contain"):
            alt_text = alt_text[len("Image may contain"):].strip()

        images_data.append({
            "url": url,
            "alt_text": alt_text,
            "model": model,
            "photographer": photographer,
            "photographer_agency": photographer_agency
        })

    return images_data

In [78]:
collection_url = "https://www.vogue.com/fashion-shows/fall-2010-ready-to-wear/oscar-de-la-renta/slideshow/collection"
images = scrape_collection_with_credits(collection_url)

for img in images:
    print(img)

No runwayShowGalleries script found.


In [79]:
import requests
import re
import json
from bs4 import BeautifulSoup

def modify_image_url(original_url):
    """Return higher resolution image URL"""
    return original_url.replace("w_360", "w_1280")

def scrape_collection(designer, show, skip_last=True):
    """
    Scrape all images from a Vogue collection page with metadata.

    Args:
        designer (str): designer name in URL format (e.g., 'oscar-de-la-renta')
        show (str): show season/year in URL format (e.g., 'fall-2010-ready-to-wear')
        skip_last (bool): whether to skip last image (designer)

    Returns:
        List of dicts: [{'url', 'alt_text', 'model', 'photographer', 'photographer_agency'}]
    """
    # Format URL
    url = f"https://www.vogue.com/fashion-shows/{show}/{designer}"
    r = requests.get(url)
    if r.status_code != 200:
        print(f"Failed to fetch {url}")
        return []

    soup = BeautifulSoup(r.content, "html.parser")

    # Find the runwayShowGalleries JSON
    script_tag = soup.find("script", string=re.compile(r'"runwayShowGalleries":'))
    if not script_tag:
        print("No runwayShowGalleries script found.")
        return []

    script_content = script_tag.string
    json_match = re.search(r'"runwayShowGalleries":\s*({.*?})\s*;', script_content, re.DOTALL)
    if not json_match:
        print("Failed to extract runwayShowGalleries JSON.")
        return []

    json_str = json_match.group(1).replace("\\u002F", "/")

    # Parse JSON safely using raw_decode
    json_decoder = json.JSONDecoder()
    try:
        gallery_json, _ = json_decoder.raw_decode(json_str)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON: {e}")
        return []

    galleries = gallery_json.get("galleries", [])
    collection_gallery = next((g for g in galleries if g.get("title") == "Collection"), None)
    if not collection_gallery:
        print("No Collection gallery found.")
        return []

    images_data = []
    items = collection_gallery["items"]
    if skip_last:
        items = items[:-1]  # skip last image

    for item in items:
        image_info = item.get("image", {})
        src = image_info.get("sources", {}).get("sm", {}).get("url")
        alt_text = image_info.get("altText")

        # Clean alt text
        if alt_text and alt_text.lower().startswith("image may contain"):
            alt_text = alt_text[len("Image may contain"):].strip()

        # Model and photographer info
        model = item.get("model")
        photographer_full = item.get("desktopCredit") or item.get("mobileCredit") or ""
        if photographer_full and "/" in photographer_full:
            photographer, photographer_agency = map(str.strip, photographer_full.split("/", 1))
        else:
            photographer = photographer_full.strip()
            photographer_agency = ""

        if src:
            images_data.append({
                "url": modify_image_url(src),
                "alt_text": alt_text,
                "model": model,
                "photographer": photographer,
                "photographer_agency": photographer_agency
            })

    return images_data


# Example usage:
designer = "oscar-de-la-renta"
show = "fall-2010-ready-to-wear"
images = scrape_collection(designer, show)

for i, img in enumerate(images, 1):
    print(f"{i}: {img['url']} | Model: {img['model']} | Photographer: {img['photographer']} ({img['photographer_agency']}) | Alt: {img['alt_text']}")

1: https://assets.vogue.com/photos/55c651ba08298d8be226dcab/master/w_1280,c_limit/00010fullscreen.jpg | Model: None | Photographer:  () | Alt: Clothing Apparel Coat Overcoat Human and Person
2: https://assets.vogue.com/photos/55c651ba08298d8be226dcac/master/w_1280,c_limit/00020fullscreen.jpg | Model: None | Photographer:  () | Alt: Clothing Apparel Human Person Sleeve Military Military Uniform Long Sleeve and Female
3: https://assets.vogue.com/photos/55c651ba08298d8be226dcad/master/w_1280,c_limit/00030fullscreen.jpg | Model: None | Photographer:  () | Alt: Clothing Sleeve Apparel Coat Long Sleeve Dress Overcoat Human Female Person and Woman
4: https://assets.vogue.com/photos/55c651ba08298d8be226dcae/master/w_1280,c_limit/00040fullscreen.jpg | Model: None | Photographer:  () | Alt: Dress Clothing Apparel Skirt Sleeve Human Female Person Long Sleeve and Woman
5: https://assets.vogue.com/photos/55c651ba08298d8be226dcaf/master/w_1280,c_limit/00050fullscreen.jpg | Model: None | Photographer

In [81]:
import requests
import json
import re
from bs4 import BeautifulSoup

def get_models_photographers(slideshow_url):
    """
    Given a slideshow URL (e.g., /slideshow/collection#1), return a list of dictionaries:
    [{'id': ..., 'model': ..., 'photographer': ..., 'photographer_agency': ...}, ...]
    """
    base_url = "https://www.vogue.com"
    if not slideshow_url.startswith("http"):
        slideshow_url = base_url + slideshow_url

    r = requests.get(slideshow_url)
    if r.status_code != 200:
        print(f"Failed to fetch slideshow: {slideshow_url}")
        return []

    soup = BeautifulSoup(r.content, "html.parser")

    # The JSON containing models & photographers is in a script tag containing "gallery"
    script_tag = soup.find("script", string=re.compile(r'"gallery"'))
    if not script_tag:
        print("No slideshow JSON script found.")
        return []

    # Extract JSON from the script
    json_match = re.search(r'"gallery":\s*({.*})\s*;', script_tag.string, re.DOTALL)
    if not json_match:
        print("Failed to extract slideshow JSON.")
        return []

    json_str = json_match.group(1).replace("\\u002F", "/")
    try:
        gallery_json = json.loads(json_str)
    except json.JSONDecodeError as e:
        print(f"Error decoding slideshow JSON: {e}")
        return []

    # Build list of images with model and photographer
    images_info = []
    for item in gallery_json.get("items", []):
        img_id = item.get("id")
        model = item.get("model")
        credit = item.get("desktopCredit") or item.get("mobileCredit") or ""
        if credit and "/" in credit:
            photographer, photographer_agency = map(str.strip, credit.split("/", 1))
        else:
            photographer = credit.strip()
            photographer_agency = ""
        images_info.append({
            "id": img_id,
            "model": model,
            "photographer": photographer,
            "photographer_agency": photographer_agency
        })

    return images_info

# Example usage
slideshow_url = "/fashion-shows/fall-2010-ready-to-wear/oscar-de-la-renta/slideshow/collection#1"
images_info = get_models_photographers(slideshow_url)
for img in images_info:
    print(img)

Failed to extract slideshow JSON.


In [73]:
import json, re

def parse_runway_show_galleries(script_content):
    images_data = []

    json_data_match = re.search(r'"runwayShowGalleries":\s*({.*?})\s*;', script_content, re.DOTALL)
    if not json_data_match:
        return images_data

    json_data_str = json_data_match.group(1).replace("\\u002F", "/")
    try:
        json_decoder = json.JSONDecoder()
        json_data, _ = json_decoder.raw_decode(json_data_str)
    except Exception as e:
        print("Error decoding JSON:", e)
        return images_data

    galleries = json_data.get("galleries", [])
    collection_gallery = next((g for g in galleries if g.get("title") == "Collection"), None)
    if not collection_gallery:
        return images_data

    for item in collection_gallery["items"][:-1]:  # skip last if it's designer
        src = item.get("image", {}).get("sources", {}).get("sm", {}).get("url")
        alt_text = item.get("image", {}).get("altText", "")
        if alt_text.lower().startswith("image may contain"):
            alt_text = alt_text[len("Image may contain"):].strip()

        slide_url = item.get("url")
        model = None
        photographer = None
        photographer_agency = None

        if slide_url:
            r = requests.get("https://www.vogue.com" + slide_url)
            if r.status_code == 200:
                soup = BeautifulSoup(r.content, "html.parser")
                slide_json_tag = soup.find("script", string=re.compile(r'"model":'))
                if slide_json_tag:
                    try:
                        slide_json = json.JSONDecoder().raw_decode(slide_json_tag.string)[0]
                        model = slide_json.get("model")
                        credit = slide_json.get("desktopCredit") or slide_json.get("mobileCredit")
                        if credit and "/" in credit:
                            photographer, photographer_agency = [c.strip() for c in credit.split("/", 1)]
                    except Exception as e:
                        print("Error decoding slide JSON:", e)

        images_data.append({
            "url": src,
            "alt_text": alt_text,
            "model": model,
            "photographer": photographer,
            "photographer_agency": photographer_agency,
            "slide_url": slide_url
        })

    return images_data

In [70]:
model

'Freja Beha Erichsen'

In [74]:
parse_runway_show_galleries(script_content)

Error decoding slide JSON: Expecting value: line 1 column 1 (char 0)
Error decoding slide JSON: Expecting value: line 1 column 1 (char 0)
Error decoding slide JSON: Expecting value: line 1 column 1 (char 0)
Error decoding slide JSON: Expecting value: line 1 column 1 (char 0)
Error decoding slide JSON: Expecting value: line 1 column 1 (char 0)
Error decoding slide JSON: Expecting value: line 1 column 1 (char 0)
Error decoding slide JSON: Expecting value: line 1 column 1 (char 0)
Error decoding slide JSON: Expecting value: line 1 column 1 (char 0)
Error decoding slide JSON: Expecting value: line 1 column 1 (char 0)
Error decoding slide JSON: Expecting value: line 1 column 1 (char 0)
Error decoding slide JSON: Expecting value: line 1 column 1 (char 0)
Error decoding slide JSON: Expecting value: line 1 column 1 (char 0)
Error decoding slide JSON: Expecting value: line 1 column 1 (char 0)
Error decoding slide JSON: Expecting value: line 1 column 1 (char 0)


KeyboardInterrupt: 

In [56]:
images_data

[{'url': 'https://assets.vogue.com/photos/55c651ba08298d8be226dcab/master/w_1280,c_limit/00010fullscreen.jpg',
  'alt_text': 'Clothing Apparel Coat Overcoat Human and Person',
  'model': 'Freja Beha Erichsen',
  'photographer': 'Marcio Madeira',
  'photographer_agency': 'FirstView.com'},
 {'url': 'https://assets.vogue.com/photos/55c651ba08298d8be226dcac/master/w_1280,c_limit/00020fullscreen.jpg',
  'alt_text': 'Clothing Apparel Human Person Sleeve Military Military Uniform Long Sleeve and Female',
  'model': 'Freja Beha Erichsen',
  'photographer': 'Marcio Madeira',
  'photographer_agency': 'FirstView.com'},
 {'url': 'https://assets.vogue.com/photos/55c651ba08298d8be226dcad/master/w_1280,c_limit/00030fullscreen.jpg',
  'alt_text': 'Clothing Sleeve Apparel Coat Long Sleeve Dress Overcoat Human Female Person and Woman',
  'model': 'Freja Beha Erichsen',
  'photographer': 'Marcio Madeira',
  'photographer_agency': 'FirstView.com'},
 {'url': 'https://assets.vogue.com/photos/55c651ba08298d8

In [49]:
images_data

[{'url': 'https://assets.vogue.com/photos/55c651ba08298d8be226dcab/master/w_1280,c_limit/00010fullscreen.jpg',
  'alt_text': 'Clothing Apparel Coat Overcoat Human and Person',
  'model': 'Freja Beha Erichsen',
  'photographer': 'Marcio Madeira',
  'photographer_agency': 'FirstView.com'},
 {'url': 'https://assets.vogue.com/photos/55c651ba08298d8be226dcac/master/w_1280,c_limit/00020fullscreen.jpg',
  'alt_text': 'Clothing Apparel Human Person Sleeve Military Military Uniform Long Sleeve and Female',
  'model': 'Freja Beha Erichsen',
  'photographer': 'Marcio Madeira',
  'photographer_agency': 'FirstView.com'},
 {'url': 'https://assets.vogue.com/photos/55c651ba08298d8be226dcad/master/w_1280,c_limit/00030fullscreen.jpg',
  'alt_text': 'Clothing Sleeve Apparel Coat Long Sleeve Dress Overcoat Human Female Person and Woman',
  'model': 'Freja Beha Erichsen',
  'photographer': 'Marcio Madeira',
  'photographer_agency': 'FirstView.com'},
 {'url': 'https://assets.vogue.com/photos/55c651ba08298d8

In [33]:
def scrape_image_page(slide_url):
    """Fetch model and photographer from individual slideshow page"""
    if not slide_url.startswith("http"):
        slide_url = "https://www.vogue.com" + slide_url

    r = requests.get(slide_url)
    if r.status_code != 200:
        return None, None

    soup = BeautifulSoup(r.content, "html.parser")

    # Find the div containing model and photo credit
    info_div = soup.find("div", class_="ImageInfoSection-ccujeY")
    if not info_div:
        return None, None

    # Extract model name
    model_tag = info_div.find("p", class_=re.compile(r"ImageInfoModel"))
    model_name = model_tag.get_text(strip=True) if model_tag else None

    # Extract photographer
    photo_tag = info_div.find("p", class_=re.compile(r"ImageInfoPhotoCredit"))
    photographer = None
    if photo_tag:
        text = photo_tag.get_text(strip=True)
        # Remove "Photo:" prefix if present
        if text.lower().startswith("photo:"):
            photographer = text[len("photo:"):].strip()
        else:
            photographer = text

    return model_name, photographer
scrape_image_page("https://www.vogue.com/fashion-shows/fall-2011-ready-to-wear/prada/slideshow/collection#40")

('Colinne Michaelis',
 'Monica Feudi / Feudiguaineri.com; video: InDigital Media')

In [37]:
import requests
from bs4 import BeautifulSoup
import re
import json
from unidecode import unidecode

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/118.0 Safari/537.36"
}

def modify_image_url(original_url):
    """Replace width parameter for higher resolution"""
    return original_url.replace("w_360", "w_1280")

def extract_creative_director(soup):
    """Extracts the creative director from the Vogue show page"""
    try:
        byline_tag = soup.find(string=re.compile(r'By\s+[A-Z][a-z]+'))
        if byline_tag:
            match = re.search(r'By\s+(.*)', byline_tag.strip())
            if match:
                creative_director = match.group(1).strip()
                creative_director = re.sub(r'[.,;]+$', '', creative_director)
                return creative_director
    except Exception as e:
        print(f"Error extracting creative director: {e}")
    return None

def scrape_image_page(slide_url):
    """Fetch model and photographer from individual slideshow page"""
    if not slide_url.startswith("http"):
        slide_url = "https://www.vogue.com" + slide_url

    r = requests.get(slide_url, headers=HEADERS)
    if r.status_code != 200:
        return None, None

    soup = BeautifulSoup(r.content, "html.parser")

    info_div = soup.find("div", class_="ImageInfoSection-ccujeY")
    if not info_div:
        return None, None

    # Model
    model_tag = info_div.find("p", class_=re.compile(r"ImageInfoModel"))
    model_name = model_tag.get_text(strip=True) if model_tag else None

    # Photographer
    photo_tag = info_div.find("p", class_=re.compile(r"ImageInfoPhotoCredit"))
    photographer = None
    if photo_tag:
        text = photo_tag.get_text(strip=True)
        if text.lower().startswith("photo:"):
            photographer = text[len("photo:"):].strip()
        else:
            photographer = text

    return model_name, photographer

def scrape_collection(url):
    """Scrape all relevant info from a Vogue collection URL"""
    r = requests.get(url, headers=HEADERS)
    if r.status_code != 200:
        print(f"Failed to fetch {url}")
        return None

    soup = BeautifulSoup(r.content, "html.parser")

    # Designer and show from URL
    match = re.search(r"/fashion-shows/(.+?)/(.+?)(?:/|$)", url)
    show = match.group(1).replace('-', ' ').title() if match else None
    designer = match.group(2).replace('-', ' ').title() if match else None

    # Description
    description_div = soup.find('div', class_='body__inner-container')
    description = description_div.get_text(strip=True) if description_div else None

    # Editor
    editor_span = soup.find('a', class_='BylineLink-gEnFiw')
    editor = editor_span.get_text(strip=True) if editor_span else None

    # Creative director
    creative_director = extract_creative_director(soup)

    # Publish date
    date_span = soup.find('time', class_='ContentHeaderPublishDate-eIBicG')
    publish_date = date_span.get_text(strip=True) if date_span else None

    # Images
    images_data = []
    try:
        script_tag = soup.find("script", string=re.compile(r'"runwayShowGalleries":'))
        if script_tag:
            json_data_match = re.search(r'"runwayShowGalleries":\s*({.*?})\s*;', script_tag.string, re.DOTALL)
            if json_data_match:
                json_data_str = json_data_match.group(1).replace("\\u002F", "/")
                json_data = json.loads(json_data_str)
                galleries = json_data.get("galleries", [])

                # Only Collection gallery
                collection_gallery = [g for g in galleries if g.get("title") == "Collection"]
                if collection_gallery:
                    items = collection_gallery[0]["items"]
                    for item in items[:-1]:  # skip last designer photo
                        image_info = item.get("image", {})
                        src = image_info.get("sources", {}).get("sm", {}).get("url")
                        alt_text = image_info.get("altText")
                        if alt_text and alt_text.lower().startswith("image may contain"):
                            alt_text = alt_text[len("Image may contain"):].strip()

                        slide_url = item.get("url")
                        model, photographer = scrape_image_page(slide_url)

                        if src:
                            images_data.append({
                                "url": modify_image_url(src),
                                "alt_text": alt_text,
                                "model": model,
                                "photographer": photographer
                            })
    except Exception as e:
        print("Error extracting images:", e)

    return {
        "designer": designer,
        "show": show,
        "description": description,
        "editor": editor,
        "publish_date": publish_date,
        "creative_director": creative_director,
        "images": images_data
    }

def extract_details_fashion_shows(fashion_string):
    """Extract location, season, year, category"""
    pattern = r'^([a-zA-Z-]+-)?(pre-)?(spring|summer|fall|winter|resort|bridal)-(\d{4})(-(menswear|ready-to-wear|couture))?$'
    match = re.match(pattern, fashion_string)
    if match:
        location = match.group(1)[:-1] if match.group(1) else ""
        if location == "":
            season = match.group(3) or match.group(2)
        else:
            season = (match.group(2) or "") + match.group(3)
        if location == 'pre':
            location = ''
            season = 'pre-fall'
        year = match.group(4)
        category = match.group(6) if match.group(6) else ""
        return location, season, year, category
    else:
        return None, None, None, None

In [39]:
url = "https://www.vogue.com/fashion-shows/fall-2011-ready-to-wear/prada"
data = scrape_collection(url)
print(data["designer"], data["show"], data["publish_date"])
print(len(data["images"]), "images collected")
print(data["images"][0])

Error extracting images: Extra data: line 1 column 1016474 (char 1016473)
Prada Fall 2011 Ready To Wear None
0 images collected


IndexError: list index out of range

In [12]:

df_school = pd.read_csv("data/names/school_names_designers_wikidata.csv")
df_school = df_school.dropna()
df_school = df_school.rename(columns= {"schoolLabel":"education"})
designer_info_df = designer_info_df.merge(df_school, how = "left")

In [27]:

designer_info_df = designer_info_df.drop_duplicates(subset= ['designer_name', 'place_of_birth', 'year_birth', 'education',
       'nationality',  'locationSchoolLabel', 'countrySchoolLabel'], ignore_index=True)


In [28]:
designer_info_df

,designer_name,place_of_birth,year_birth,education,nationality,employer,locationSchoolLabel,countrySchoolLabel
0,Aage Thaarup,Copenhagen,1906.0,None,Denmark,[],None,None
1,Achille Maramotti,Reggio Emilia,1927.0,Sapienza University of Rome,Italy,[Max Mara],Gaeta,Italy
2,Achille Maramotti,Reggio Emilia,1927.0,Sapienza University of Rome,Italy,[Max Mara],Guidonia Montecelio,Italy
3,Achille Maramotti,Reggio Emilia,1927.0,Sapienza University of Rome,Italy,[Max Mara],Isernia,Italy
4,Achille Maramotti,Reggio Emilia,1927.0,Sapienza University of Rome,Italy,[Max Mara],Pomezia,Italy
...,...,...,...,...,...,...,...,...
797,Zandra Rhodes,Chatham,1940.0,University for the Creative Arts,United Kingdom,None,None,None
798,Zandra Rhodes,Kent,1961.0,None,United Kingdom,[],None,None
799,Zoe Latta,None,NaN,Rhode Island School of Design,None,None,Providence,United States
800,Zowie Broach,None,NaN,None,None,None,None,None


In [29]:
designer_info_df.to_parquet("data/final_info_designers.parquet")